## SRP008012

**paper:** [PMID: 22499666](https://pmc.ncbi.nlm.nih.gov/articles/PMC3396367/) - Disentangling the relationship between sex-biased gene expression and X-linkage

**date, curator:** 2026-06-22, Sara Carsanaro

**notes**
* protocol not clear but this is definitely polyA


### annotation summary

In [32]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,Abdomen,UBERON:6003023,insect adult abdomen,perfect match
2,Thorax,UBERON:6003018,insect adult thorax,perfect match
4,Head,UBERON:6003007,insect adult head,perfect match


In [33]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,5-7 days post eclosion,UBERON:0000066,fully formed stage,perfect match


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP008012"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
22-Jun-2026 16:01:53 DEBUG utils - Directory ./ already exists. Skipping.
22-Jun-2026 16:01:53 INFO GEOparse - Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE31nnn/GSE31723/soft/GSE31723_family.soft.gz to ./GSE31723_family.soft.gz
100%|██████████████████████████████████████| 2.10k/2.10k [00:01<00:00, 1.93kB/s]
22-Jun-2026 16:01:56 DEBUG downloader - Size validation passed
22-Jun-2026 16:01:56 DEBUG downloader - Moving /var/folders/b5/crkp117d43q5mcndnwlrww3w0000gn/T/tmpfu46ednh to /Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/SRP008012/GSE31723_family.soft.gz
22-Jun-2026 16

### library annnotations

In [11]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX095362,SRP008012,Illumina HiSeq 2000,SRS259360,,,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM787644,Abdomen,5-7 days post eclosion,,,,M,,,7260,,,,,,D. willistoni male abdomen rep1,"SAMN00715056,GSM787644",,,,,,,,,,,22/06/2026,"Whole flies were placed live in Ringer's solution. Heads, thoraxes, and abdomens were dissected. Samples from each segment were flash frozen on dry ice and stored at -80C. RNA was extracted from the samples using TRIzol (Invitrogen Inc, Carlsbad, CA) following the manufacturers instructions. Poly-adenylated RNA was isolated using oligo-dT Dynabeads (Invitrogen Inc, Carlsbad, CA), mRNA was fragmented using alkaline hydrolysis, double-stranded cDNA was created using random hexamer primers, sequencing adapters were ligated to the cDNA, fragments were size selected by gel purification, and the size selected fragments were amplified by PCR.",,GSM787644,,Abdomen,,Adult,,TRANSCRIPTOMIC,cDNA
1,SRX095361,SRP008012,Illumina HiSeq 2000,SRS259359,,,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM787643,Abdomen,5-7 days post eclosion,,,,F,,,7260,,,,,,D. willistoni female abdomen rep1,"SAMN00715055,GSM787643",,,,,,,,,,,22/06/2026,"Whole flies were placed live in Ringer's solution. Heads, thoraxes, and abdomens were dissected. Samples from each segment were flash frozen on dry ice and stored at -80C. RNA was extracted from the samples using TRIzol (Invitrogen Inc, Carlsbad, CA) following the manufacturers instructions. Poly-adenylated RNA was isolated using oligo-dT Dynabeads (Invitrogen Inc, Carlsbad, CA), mRNA was fragmented using alkaline hydrolysis, double-stranded cDNA was created using random hexamer primers, sequencing adapters were ligated to the cDNA, fragments were size selected by gel purification, and the size selected fragments were amplified by PCR.",,GSM787643,,Abdomen,,Adult,,TRANSCRIPTOMIC,cDNA
2,SRX095360,SRP008012,Illumina Genome Analyzer II,SRS259358,,,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM787642,Thorax,5-7 days post eclosion,,,,M,,,7260,,,,,,D. willistoni male thorax rep1,"SAMN00715054,GSM787642",,,,,,,,,,,22/06/2026,"Whole flies were placed live in Ringer's solution. Heads, thoraxes, and abdomens were dissected. Samples from each segment were flash frozen on dry ice and stored at -80C. RNA was extracted from the samples using TRIzol (Invitrogen Inc, Carlsbad, CA) following the manufacturers instructions. Poly-adenylated RNA was isolated using oligo-dT Dynabeads (Invitrogen Inc, Carlsbad, CA), mRNA was fragmented using alkaline hydrolysis, double-stranded cDNA was created using random hexamer primers, sequencing adapters were ligated to the cDNA, fragments were size selected by gel purification, and the size selected fragments were amplified by PCR.",,GSM787642,,Thorax,,Adult,,TRANSCRIPTOMIC,cDNA
3,SRX095359,SRP008012,Illumina Genome Analyzer II,SRS259357,,,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM787641,Thorax,5-7 days post eclosion,,,,F,,,7260,,,,,,D. willistoni female thorax rep1,"SAMN00715053,GSM787641",,,,,,,,,,,22/06/2026,"Whole flies were placed live in Ringer's solution. Heads, thoraxes, and abdomens were dissected. Samples from each segment were flash frozen on dry ice and stored at -80C. RNA was extracted from the samples using TRIzol (Invitrogen Inc, Carlsbad, CA) following the manufacturers instructions. Poly-adenylated RNA was isolated using oligo-dT Dynabeads (Invitrogen Inc, Carlsbad, CA), 

#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [12]:
unique_sorted(library, "infoOrgan")

['Abdomen' 'Head' 'Thorax']


In [13]:

# Abdomen
library.loc[library["infoOrgan"] == "Abdomen", "anatId"] = "UBERON:6003023"
library.loc[library["infoOrgan"] == "Abdomen", "anatName"] = "insect adult abdomen"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "Abdomen", "anatAnnotationStatus"] = "perfect match"
# full sampling, partial sampling, not documented
library.loc[library["infoOrgan"] == "Abdomen", "anatBiologicalStatus"] = "not documented"

# Thorax
library.loc[library["infoOrgan"] == "Thorax", "anatId"] = "UBERON:6003018"
library.loc[library["infoOrgan"] == "Thorax", "anatName"] = "insect adult thorax"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "Thorax", "anatAnnotationStatus"] = "perfect match"
# full sampling, partial sampling, not documented
library.loc[library["infoOrgan"] == "Thorax", "anatBiologicalStatus"] = "not documented"

# Head
library.loc[library["infoOrgan"] == "Head", "anatId"] = "UBERON:6003007"
library.loc[library["infoOrgan"] == "Head", "anatName"] = "insect adult head"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "Head", "anatAnnotationStatus"] = "perfect match"
# full sampling, partial sampling, not documented
library.loc[library["infoOrgan"] == "Head", "anatBiologicalStatus"] = "not documented"

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX095362,SRP008012,Illumina HiSeq 2000,SRS259360,UBERON:6003023,insect adult abdomen,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM787644,Abdomen,5-7 days post eclosion,perfect match,not documented,,M,,,7260,,,,,,D. willistoni male abdomen rep1,"SAMN00715056,GSM787644",,,,,,,,,,,22/06/2026,"Whole flies were placed live in Ringer's solution. Heads, thoraxes, and abdomens were dissected. Samples from each segment were flash frozen on dry ice and stored at -80C. RNA was extracted from the samples using TRIzol (Invitrogen Inc, Carlsbad, CA) following the manufacturers instructions. Poly-adenylated RNA was isolated using oligo-dT Dynabeads (Invitrogen Inc, Carlsbad, CA), mRNA was fragmented using alkaline hydrolysis, double-stranded cDNA was created using random hexamer primers, sequencing adapters were ligated to the cDNA, fragments were size selected by gel purification, and the size selected fragments were amplified by PCR.",,GSM787644,,Abdomen,,Adult,,TRANSCRIPTOMIC,cDNA
1,SRX095361,SRP008012,Illumina HiSeq 2000,SRS259359,UBERON:6003023,insect adult abdomen,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM787643,Abdomen,5-7 days post eclosion,perfect match,not documented,,F,,,7260,,,,,,D. willistoni female abdomen rep1,"SAMN00715055,GSM787643",,,,,,,,,,,22/06/2026,"Whole flies were placed live in Ringer's solution. Heads, thoraxes, and abdomens were dissected. Samples from each segment were flash frozen on dry ice and stored at -80C. RNA was extracted from the samples using TRIzol (Invitrogen Inc, Carlsbad, CA) following the manufacturers instructions. Poly-adenylated RNA was isolated using oligo-dT Dynabeads (Invitrogen Inc, Carlsbad, CA), mRNA was fragmented using alkaline hydrolysis, double-stranded cDNA was created using random hexamer primers, sequencing adapters were ligated to the cDNA, fragments were size selected by gel purification, and the size selected fragments were amplified by PCR.",,GSM787643,,Abdomen,,Adult,,TRANSCRIPTOMIC,cDNA
2,SRX095360,SRP008012,Illumina Genome Analyzer II,SRS259358,UBERON:6003018,insect adult thorax,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM787642,Thorax,5-7 days post eclosion,perfect match,not documented,,M,,,7260,,,,,,D. willistoni male thorax rep1,"SAMN00715054,GSM787642",,,,,,,,,,,22/06/2026,"Whole flies were placed live in Ringer's solution. Heads, thoraxes, and abdomens were dissected. Samples from each segment were flash frozen on dry ice and stored at -80C. RNA was extracted from the samples using TRIzol (Invitrogen Inc, Carlsbad, CA) following the manufacturers instructions. Poly-adenylated RNA was isolated using oligo-dT Dynabeads (Invitrogen Inc, Carlsbad, CA), mRNA was fragmented using alkaline hydrolysis, double-stranded cDNA was created using random hexamer primers, sequencing adapters were ligated to the cDNA, fragments were size selected by gel purification, and the size selected fragments were amplified by PCR.",,GSM787642,,Thorax,,Adult,,TRANSCRIPTOMIC,cDNA
3,SRX095359,SRP008012,Illumina Genome Analyzer II,SRS259357,UBERON:6003018,insect adult thorax,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM787641,Thorax,5-7 days post eclosion,perfect match,not documented,,F,,,7260,,,,,,D. willistoni female thorax rep1,"SAMN00715053,GSM787641",,,,,,,,,,,22/06/2026,"Whole flies were placed live in Ringer's solution. Heads, thoraxes, and abdomens were dissected. Samples from each segment were flash frozen 

#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [14]:
unique_sorted(library, "infoStage")

['5-7 days post eclosion']


In [15]:
# all
library.loc[:,'stageId'] = 'UBERON:0000066'
library.loc[:,'stageName'] = 'fully formed stage'
# perfect match, missing child term, other
library.loc[:,'stageAnnotationStatus'] = 'perfect match'


# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX095362,SRP008012,Illumina HiSeq 2000,SRS259360,UBERON:6003023,insect adult abdomen,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM787644,Abdomen,5-7 days post eclosion,perfect match,not documented,perfect match,M,,,7260,,,,,,D. willistoni male abdomen rep1,"SAMN00715056,GSM787644",,,,,,,,,,,22/06/2026,"Whole flies were placed live in Ringer's solution. Heads, thoraxes, and abdomens were dissected. Samples from each segment were flash frozen on dry ice and stored at -80C. RNA was extracted from the samples using TRIzol (Invitrogen Inc, Carlsbad, CA) following the manufacturers instructions. Poly-adenylated RNA was isolated using oligo-dT Dynabeads (Invitrogen Inc, Carlsbad, CA), mRNA was fragmented using alkaline hydrolysis, double-stranded cDNA was created using random hexamer primers, sequencing adapters were ligated to the cDNA, fragments were size selected by gel purification, and the size selected fragments were amplified by PCR.",,GSM787644,,Abdomen,,Adult,,TRANSCRIPTOMIC,cDNA
1,SRX095361,SRP008012,Illumina HiSeq 2000,SRS259359,UBERON:6003023,insect adult abdomen,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM787643,Abdomen,5-7 days post eclosion,perfect match,not documented,perfect match,F,,,7260,,,,,,D. willistoni female abdomen rep1,"SAMN00715055,GSM787643",,,,,,,,,,,22/06/2026,"Whole flies were placed live in Ringer's solution. Heads, thoraxes, and abdomens were dissected. Samples from each segment were flash frozen on dry ice and stored at -80C. RNA was extracted from the samples using TRIzol (Invitrogen Inc, Carlsbad, CA) following the manufacturers instructions. Poly-adenylated RNA was isolated using oligo-dT Dynabeads (Invitrogen Inc, Carlsbad, CA), mRNA was fragmented using alkaline hydrolysis, double-stranded cDNA was created using random hexamer primers, sequencing adapters were ligated to the cDNA, fragments were size selected by gel purification, and the size selected fragments were amplified by PCR.",,GSM787643,,Abdomen,,Adult,,TRANSCRIPTOMIC,cDNA
2,SRX095360,SRP008012,Illumina Genome Analyzer II,SRS259358,UBERON:6003018,insect adult thorax,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM787642,Thorax,5-7 days post eclosion,perfect match,not documented,perfect match,M,,,7260,,,,,,D. willistoni male thorax rep1,"SAMN00715054,GSM787642",,,,,,,,,,,22/06/2026,"Whole flies were placed live in Ringer's solution. Heads, thoraxes, and abdomens were dissected. Samples from each segment were flash frozen on dry ice and stored at -80C. RNA was extracted from the samples using TRIzol (Invitrogen Inc, Carlsbad, CA) following the manufacturers instructions. Poly-adenylated RNA was isolated using oligo-dT Dynabeads (Invitrogen Inc, Carlsbad, CA), mRNA was fragmented using alkaline hydrolysis, double-stranded cDNA was created using random hexamer primers, sequencing adapters were ligated to the cDNA, fragments were size selected by gel purification, and the size selected fragments were amplified by PCR.",,GSM787642,,Thorax,,Adult,,TRANSCRIPTOMIC,cDNA
3,SRX095359,SRP008012,Illumina Genome Analyzer II,SRS259357,UBERON:6003018,insect adult thorax,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM787641,Thorax,5-7 days post eclosion,perfect match,not documented,perfect match,F,,,7260,,,,,,D. willistoni female thorax rep1,"SAMN007

#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [ ]:
#library.loc[:,'sex'] = ''
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [16]:
# making these variables because we use them again in the experiment file
#my_protocol = ''
# full_length or 3'
#my_protocolType = ''

#library.loc[:,'protocol'] = my_protocol
#library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX095362,SRP008012,Illumina HiSeq 2000,SRS259360,UBERON:6003023,insect adult abdomen,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM787644,Abdomen,5-7 days post eclosion,perfect match,not documented,perfect match,M,,,7260,,,polyA,,,D. willistoni male abdomen rep1,"SAMN00715056,GSM787644",,,,,,,,,,,22/06/2026,"Whole flies were placed live in Ringer's solution. Heads, thoraxes, and abdomens were dissected. Samples from each segment were flash frozen on dry ice and stored at -80C. RNA was extracted from the samples using TRIzol (Invitrogen Inc, Carlsbad, CA) following the manufacturers instructions. Poly-adenylated RNA was isolated using oligo-dT Dynabeads (Invitrogen Inc, Carlsbad, CA), mRNA was fragmented using alkaline hydrolysis, double-stranded cDNA was created using random hexamer primers, sequencing adapters were ligated to the cDNA, fragments were size selected by gel purification, and the size selected fragments were amplified by PCR.",,GSM787644,,Abdomen,,Adult,,TRANSCRIPTOMIC,cDNA
1,SRX095361,SRP008012,Illumina HiSeq 2000,SRS259359,UBERON:6003023,insect adult abdomen,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM787643,Abdomen,5-7 days post eclosion,perfect match,not documented,perfect match,F,,,7260,,,polyA,,,D. willistoni female abdomen rep1,"SAMN00715055,GSM787643",,,,,,,,,,,22/06/2026,"Whole flies were placed live in Ringer's solution. Heads, thoraxes, and abdomens were dissected. Samples from each segment were flash frozen on dry ice and stored at -80C. RNA was extracted from the samples using TRIzol (Invitrogen Inc, Carlsbad, CA) following the manufacturers instructions. Poly-adenylated RNA was isolated using oligo-dT Dynabeads (Invitrogen Inc, Carlsbad, CA), mRNA was fragmented using alkaline hydrolysis, double-stranded cDNA was created using random hexamer primers, sequencing adapters were ligated to the cDNA, fragments were size selected by gel purification, and the size selected fragments were amplified by PCR.",,GSM787643,,Abdomen,,Adult,,TRANSCRIPTOMIC,cDNA
2,SRX095360,SRP008012,Illumina Genome Analyzer II,SRS259358,UBERON:6003018,insect adult thorax,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM787642,Thorax,5-7 days post eclosion,perfect match,not documented,perfect match,M,,,7260,,,polyA,,,D. willistoni male thorax rep1,"SAMN00715054,GSM787642",,,,,,,,,,,22/06/2026,"Whole flies were placed live in Ringer's solution. Heads, thoraxes, and abdomens were dissected. Samples from each segment were flash frozen on dry ice and stored at -80C. RNA was extracted from the samples using TRIzol (Invitrogen Inc, Carlsbad, CA) following the manufacturers instructions. Poly-adenylated RNA was isolated using oligo-dT Dynabeads (Invitrogen Inc, Carlsbad, CA), mRNA was fragmented using alkaline hydrolysis, double-stranded cDNA was created using random hexamer primers, sequencing adapters were ligated to the cDNA, fragments were size selected by gel purification, and the size selected fragments were amplified by PCR.",,GSM787642,,Thorax,,Adult,,TRANSCRIPTOMIC,cDNA
3,SRX095359,SRP008012,Illumina Genome Analyzer II,SRS259357,UBERON:6003018,insect adult thorax,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM787641,Thorax,5-7 days post eclosion,perfect match,not documented,perfect match,F,,,7260,,,polyA,,,D. willistoni female 

#### globin, replicates

In [17]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [21]:
library.loc[:,'sampleAge_value'] = '5-7'
library.loc[:,'sampleAge_unit'] = 'day post eclosion'

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX095362,SRP008012,Illumina HiSeq 2000,SRS259360,UBERON:6003023,insect adult abdomen,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM787644,Abdomen,5-7 days post eclosion,perfect match,not documented,perfect match,M,,,7260,,,polyA,,,D. willistoni male abdomen rep1,"SAMN00715056,GSM787644",5-7,day post eclosion,,,,,,,,SAC,2026-06-25,"Whole flies were placed live in Ringer's solution. Heads, thoraxes, and abdomens were dissected. Samples from each segment were flash frozen on dry ice and stored at -80C. RNA was extracted from the samples using TRIzol (Invitrogen Inc, Carlsbad, CA) following the manufacturers instructions. Poly-adenylated RNA was isolated using oligo-dT Dynabeads (Invitrogen Inc, Carlsbad, CA), mRNA was fragmented using alkaline hydrolysis, double-stranded cDNA was created using random hexamer primers, sequencing adapters were ligated to the cDNA, fragments were size selected by gel purification, and the size selected fragments were amplified by PCR.",,GSM787644,,Abdomen,,Adult,,TRANSCRIPTOMIC,cDNA
1,SRX095361,SRP008012,Illumina HiSeq 2000,SRS259359,UBERON:6003023,insect adult abdomen,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM787643,Abdomen,5-7 days post eclosion,perfect match,not documented,perfect match,F,,,7260,,,polyA,,,D. willistoni female abdomen rep1,"SAMN00715055,GSM787643",5-7,day post eclosion,,,,,,,,SAC,2026-06-25,"Whole flies were placed live in Ringer's solution. Heads, thoraxes, and abdomens were dissected. Samples from each segment were flash frozen on dry ice and stored at -80C. RNA was extracted from the samples using TRIzol (Invitrogen Inc, Carlsbad, CA) following the manufacturers instructions. Poly-adenylated RNA was isolated using oligo-dT Dynabeads (Invitrogen Inc, Carlsbad, CA), mRNA was fragmented using alkaline hydrolysis, double-stranded cDNA was created using random hexamer primers, sequencing adapters were ligated to the cDNA, fragments were size selected by gel purification, and the size selected fragments were amplified by PCR.",,GSM787643,,Abdomen,,Adult,,TRANSCRIPTOMIC,cDNA
2,SRX095360,SRP008012,Illumina Genome Analyzer II,SRS259358,UBERON:6003018,insect adult thorax,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM787642,Thorax,5-7 days post eclosion,perfect match,not documented,perfect match,M,,,7260,,,polyA,,,D. willistoni male thorax rep1,"SAMN00715054,GSM787642",5-7,day post eclosion,,,,,,,,SAC,2026-06-25,"Whole flies were placed live in Ringer's solution. Heads, thoraxes, and abdomens were dissected. Samples from each segment were flash frozen on dry ice and stored at -80C. RNA was extracted from the samples using TRIzol (Invitrogen Inc, Carlsbad, CA) following the manufacturers instructions. Poly-adenylated RNA was isolated using oligo-dT Dynabeads (Invitrogen Inc, Carlsbad, CA), mRNA was fragmented using alkaline hydrolysis, double-stranded cDNA was created using random hexamer primers, sequencing adapters were ligated to the cDNA, fragments were size selected by gel purification, and the size selected fragments were amplified by PCR.",,GSM787642,,Thorax,,Adult,,TRANSCRIPTOMIC,cDNA
3,SRX095359,SRP008012,Illumina Genome Analyzer II,SRS259357,UBERON:6003018,insect adult thorax,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM787641,Thorax,5-7 days post eclosion,perfect match,

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [22]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-06-25'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX095362,SRP008012,Illumina HiSeq 2000,SRS259360,UBERON:6003023,insect adult abdomen,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM787644,Abdomen,5-7 days post eclosion,perfect match,not documented,perfect match,M,,,7260,,,polyA,,,D. willistoni male abdomen rep1,"SAMN00715056,GSM787644",5-7,day post eclosion,,,,,,,,SAC,2026-06-25,"Whole flies were placed live in Ringer's solution. Heads, thoraxes, and abdomens were dissected. Samples from each segment were flash frozen on dry ice and stored at -80C. RNA was extracted from the samples using TRIzol (Invitrogen Inc, Carlsbad, CA) following the manufacturers instructions. Poly-adenylated RNA was isolated using oligo-dT Dynabeads (Invitrogen Inc, Carlsbad, CA), mRNA was fragmented using alkaline hydrolysis, double-stranded cDNA was created using random hexamer primers, sequencing adapters were ligated to the cDNA, fragments were size selected by gel purification, and the size selected fragments were amplified by PCR.",,GSM787644,,Abdomen,,Adult,,TRANSCRIPTOMIC,cDNA
1,SRX095361,SRP008012,Illumina HiSeq 2000,SRS259359,UBERON:6003023,insect adult abdomen,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM787643,Abdomen,5-7 days post eclosion,perfect match,not documented,perfect match,F,,,7260,,,polyA,,,D. willistoni female abdomen rep1,"SAMN00715055,GSM787643",5-7,day post eclosion,,,,,,,,SAC,2026-06-25,"Whole flies were placed live in Ringer's solution. Heads, thoraxes, and abdomens were dissected. Samples from each segment were flash frozen on dry ice and stored at -80C. RNA was extracted from the samples using TRIzol (Invitrogen Inc, Carlsbad, CA) following the manufacturers instructions. Poly-adenylated RNA was isolated using oligo-dT Dynabeads (Invitrogen Inc, Carlsbad, CA), mRNA was fragmented using alkaline hydrolysis, double-stranded cDNA was created using random hexamer primers, sequencing adapters were ligated to the cDNA, fragments were size selected by gel purification, and the size selected fragments were amplified by PCR.",,GSM787643,,Abdomen,,Adult,,TRANSCRIPTOMIC,cDNA
2,SRX095360,SRP008012,Illumina Genome Analyzer II,SRS259358,UBERON:6003018,insect adult thorax,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM787642,Thorax,5-7 days post eclosion,perfect match,not documented,perfect match,M,,,7260,,,polyA,,,D. willistoni male thorax rep1,"SAMN00715054,GSM787642",5-7,day post eclosion,,,,,,,,SAC,2026-06-25,"Whole flies were placed live in Ringer's solution. Heads, thoraxes, and abdomens were dissected. Samples from each segment were flash frozen on dry ice and stored at -80C. RNA was extracted from the samples using TRIzol (Invitrogen Inc, Carlsbad, CA) following the manufacturers instructions. Poly-adenylated RNA was isolated using oligo-dT Dynabeads (Invitrogen Inc, Carlsbad, CA), mRNA was fragmented using alkaline hydrolysis, double-stranded cDNA was created using random hexamer primers, sequencing adapters were ligated to the cDNA, fragments were size selected by gel purification, and the size selected fragments were amplified by PCR.",,GSM787642,,Thorax,,Adult,,TRANSCRIPTOMIC,cDNA
3,SRX095359,SRP008012,Illumina Genome Analyzer II,SRS259357,UBERON:6003018,insect adult thorax,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM787641,Thorax,5-7 days post eclosion,perfect match,

#### comments

In [23]:
library.loc[:,'comment'] = 'PMID: 22499666'

#### save complete file with correct columns

In [24]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX095362,SRP008012,Illumina HiSeq 2000,SRS259360,UBERON:6003023,insect adult abdomen,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM787644,Abdomen,5-7 days post eclosion,perfect match,not documented,perfect match,M,,,7260,,,polyA,,,D. willistoni male abdomen rep1,"SAMN00715056,GSM787644",5-7,day post eclosion,,,,,PMID: 22499666,,,SAC,2026-06-25
1,SRX095361,SRP008012,Illumina HiSeq 2000,SRS259359,UBERON:6003023,insect adult abdomen,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM787643,Abdomen,5-7 days post eclosion,perfect match,not documented,perfect match,F,,,7260,,,polyA,,,D. willistoni female abdomen rep1,"SAMN00715055,GSM787643",5-7,day post eclosion,,,,,PMID: 22499666,,,SAC,2026-06-25
2,SRX095360,SRP008012,Illumina Genome Analyzer II,SRS259358,UBERON:6003018,insect adult thorax,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM787642,Thorax,5-7 days post eclosion,perfect match,not documented,perfect match,M,,,7260,,,polyA,,,D. willistoni male thorax rep1,"SAMN00715054,GSM787642",5-7,day post eclosion,,,,,PMID: 22499666,,,SAC,2026-06-25
3,SRX095359,SRP008012,Illumina Genome Analyzer II,SRS259357,UBERON:6003018,insect adult thorax,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM787641,Thorax,5-7 days post eclosion,perfect match,not documented,perfect match,F,,,7260,,,polyA,,,D. willistoni female thorax rep1,"SAMN00715053,GSM787641",5-7,day post eclosion,,,,,PMID: 22499666,,,SAC,2026-06-25
4,SRX095358,SRP008012,Illumina HiSeq 2000,SRS259356,UBERON:6003007,insect adult head,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM787640,Head,5-7 days post eclosion,perfect match,not documented,perfect match,M,,,7260,,,polyA,,,D. willistoni male head rep2,"SAMN00715052,GSM787640",5-7,day post eclosion,,,,,PMID: 22499666,,,SAC,2026-06-25
5,SRX095357,SRP008012,Illumina HiSeq 2000,SRS259355,UBERON:6003007,insect adult head,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM787639,Head,5-7 days post eclosion,perfect match,not documented,perfect match,F,,,7260,,,polyA,,,D. willistoni female head rep2,"SAMN00715051,GSM787639",5-7,day post eclosion,,,,,PMID: 22499666,,,SAC,2026-06-25
6,SRX095356,SRP008012,Illumina Genome Analyzer II,SRS259354,UBERON:6003007,insect adult head,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM787638,Head,5-7 days post eclosion,perfect match,not documented,perfect match,M,,,7260,,,polyA,,,D. willistoni male head rep1,"SAMN00715050,GSM787638",5-7,day post eclosion,,,,,PMID: 22499666,,,SAC,2026-06-25
7,SRX095355,SRP008012,Illumina Genome Analyzer II,SRS259353,UBERON:6003007,insect adult head,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM787637,Head,5-7 days post eclosion,perfect match,not documented,perfect match,F,,,7260,,,polyA,,,D. willistoni female head rep1,"SAMN00715049,GSM787637",5-7,day post eclosion,,,,,PMID: 22499666,,,SAC,2026-06-25


### experiment annotations

In [25]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP008012,Sex-biased expression in Drosophila willistoni,"We performed mRNA sequencing of samples isolated from the heads, thoraxes, and abdomens of males and females of Drosophila willistoni to identify genes that are differentially expressed between the sexes. Overall design: Comparison of expression levels in females and males",SRA,,,,,,GSE31723,PRJNA145349,22499666,,10.1101/gr.132100.111,,


#### experiment and protocol details

In [26]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

8

In [27]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'total'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
#experiment.loc[:,'protocol'] = my_protocol
#experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP008012,Sex-biased expression in Drosophila willistoni,"We performed mRNA sequencing of samples isolated from the heads, thoraxes, and abdomens of males and females of Drosophila willistoni to identify genes that are differentially expressed between the sexes. Overall design: Comparison of expression levels in females and males",SRA,total,Bgee 1K,8,,,GSE31723,PRJNA145349,22499666,,10.1101/gr.132100.111,,


#### paper and xrefs

In [28]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '22499666'
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC3396367/'
experiment.loc[:,'DOI'] = '10.1101/gr.132100.111'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP008012,Sex-biased expression in Drosophila willistoni,"We performed mRNA sequencing of samples isolated from the heads, thoraxes, and abdomens of males and females of Drosophila willistoni to identify genes that are differentially expressed between the sexes. Overall design: Comparison of expression levels in females and males",SRA,total,Bgee 1K,8,,,GSE31723,PRJNA145349,22499666,https://pmc.ncbi.nlm.nih.gov/articles/PMC3396367/,10.1101/gr.132100.111,,


#### comments

In [ ]:
#experiment.loc[:,'comment'] = ''

display_df(experiment)

#### save complete file

In [29]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [30]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [31]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 8
Errors: 8
Warnings: 0
Top codes:
  - BAD_VALUE: 8


#### check columns match

In [34]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [35]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
66128,SRX660277,SRP044761,Illumina HiSeq 2000,SRS665276,FBbt:00001766,eye-antennal disc,FBdv:00005339,third instar larval stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,eye-antennal imaginal disc,3rd instar larvae,perfect match,not documented,perfect match,NA,,Canton-S,7227,,,polyA,,,RNA_Dmel_CS_EA: RNA_seq_eye-antennal_imaginal_...,"SAMN02934561,GSM1444010",,,,,,,,,,SAC,2026-06-25
66129,SRX660276,SRP044761,Illumina HiSeq 2000,SRS665275,FBbt:00001766,eye-antennal disc,FBdv:00005339,third instar larval stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,eye-antennal imaginal disc,3rd instar larvae,perfect match,not documented,perfect match,NA,,DGRP-208,7227,,,polyA,,,RNA_Dmel_RAL-208_EA: RNA_seq_eye-antennal_imag...,"SAMN02934560,GSM1444009",,,,,,,,,,SAC,2026-06-25
66130,SRX095362,SRP008012,Illumina HiSeq 2000,SRS259360,UBERON:6003023,insect adult abdomen,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?...,Abdomen,5-7 days post eclosion,perfect match,not documented,perfect match,M,,,7260,,,polyA,,,D. willistoni male abdomen rep1,"SAMN00715056,GSM787644",5-7,day post eclosion,,,,,PMID: 22499666,,,SAC,2026-06-25
66131,SRX095361,SRP008012,Illumina HiSeq 2000,SRS259359,UBERON:6003023,insect adult abdomen,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?...,Abdomen,5-7 days post eclosion,perfect match,not documented,perfect match,F,,,7260,,,polyA,,,D. willistoni female abdomen rep1,"SAMN00715055,GSM787643",5-7,day post eclosion,,,,,PMID: 22499666,,,SAC,2026-06-25
66132,SRX095360,SRP008012,Illumina Genome Analyzer II,SRS259358,UBERON:6003018,insect adult thorax,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?...,Thorax,5-7 days post eclosion,perfect match,not documented,perfect match,M,,,7260,,,polyA,,,D. willistoni male thorax rep1,"SAMN00715054,GSM787642",5-7,day post eclosion,,,,,PMID: 22499666,,,SAC,2026-06-25
66133,SRX095359,SRP008012,Illumina Genome Analyzer II,SRS259357,UBERON:6003018,insect adult thorax,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?...,Thorax,5-7 days post eclosion,perfect match,not documented,perfect match,F,,,7260,,,polyA,,,D. willistoni female thorax rep1,"SAMN00715053,GSM787641",5-7,day post eclosion,,,,,PMID: 22499666,,,SAC,2026-06-25
66134,SRX095358,SRP008012,Illumina HiSeq 2000,SRS259356,UBERON:6003007,insect adult head,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?...,Head,5-7 days post eclosion,perfect match,not documented,perfect match,M,,,7260,,,polyA,,,D. willistoni male head rep2,"SAMN00715052,GSM787640",5-7,day post eclosion,,,,,PMID: 22499666,,,SAC,2026-06-25


In [36]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1268,SRP582736,Evolution of limb and digit identity genes sin...,The datasets are used to create the analysis o...,SRA,total,Bgee 1K,249,"NEBNext Ultra RNA Library Prep Kit,DNBSEQ-T7",full_length,,PRJNA1258050,41982182,https://pmc.ncbi.nlm.nih.gov/articles/PMC13135...,10.1093/molbev/msag101,,contains DNBSEQ-T7 libraries that should maybe...
1269,SRP044761,Evolutionary changes in cis regulation are ass...,Expression profiling by high-throughput sequen...,SRA,total,Bgee 1K,27,,,GSE59707,PRJNA256000,,https://kuleuven.limo.libis.be/discovery/fulld...,,,"no paper, seems likely this is part of a thesi..."
1270,SRP008012,Sex-biased expression in Drosophila willistoni,We performed mRNA sequencing of samples isolat...,SRA,total,Bgee 1K,8,,,GSE31723,PRJNA145349,22499666,https://pmc.ncbi.nlm.nih.gov/articles/PMC3396367/,10.1101/gr.132100.111,,


### add annotations to git

In [37]:
! git pull

Already up to date.


In [38]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [ ]:
! git status

In [39]:
! git add $git_experiment_path $git_library_path

In [40]:
! git commit -m $commit_message_exp

[develop 5b78c63] adding annotated bulk experiment SRP008012
 2 files changed, 9 insertions(+)


In [41]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 2.33 KiB | 2.33 MiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   99bc727..5b78c63  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push